# Generate inflation_multipliers.csv

This notebook reads `FredCPI.csv`, uses the December CPI value for each year, and creates `inflation_multipliers.csv` with multipliers relative to December 1997.

In [1]:
from pathlib import Path

import pandas as pd

data_dir = Path.cwd()
input_path = data_dir / "FredCPI.csv"
output_path = data_dir / "inflation_multipliers.csv"
base_year = 1997

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

raw = pd.read_csv(input_path)
if raw.empty:
    raise ValueError("FredCPI.csv is empty.")

date_col = next(
    (col for col in raw.columns if col.strip().lower() in {"date", "observation_date"}),
    raw.columns[0],
)
value_cols = [col for col in raw.columns if col != date_col]
if not value_cols:
    raise ValueError("Could not identify the CPI value column in FredCPI.csv.")
value_col = value_cols[0]

df = raw[[date_col, value_col]].rename(columns={date_col: "date", value_col: "cpi"}).copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["cpi"] = pd.to_numeric(df["cpi"], errors="coerce")
df = df.dropna(subset=["date", "cpi"]).sort_values("date")

december = df[df["date"].dt.month == 12].copy()
if december.empty:
    raise ValueError("No December observations were found in FredCPI.csv.")

december["year"] = december["date"].dt.year
december = december.sort_values("date").drop_duplicates(subset="year", keep="last")
december = december[december["year"] >= base_year].copy()

base_rows = december.loc[december["year"] == base_year, "cpi"]
if base_rows.empty:
    raise ValueError(f"December {base_year} CPI value was not found in FredCPI.csv.")
base_cpi = float(base_rows.iloc[0])

result = december[["year", "cpi"]].copy()
result["inflation_multiplier"] = (result["cpi"] / base_cpi).round(2)
result = result[["year", "inflation_multiplier"]].reset_index(drop=True)

result.to_csv(output_path, index=False)
result.head(), output_path

(   year  inflation_multiplier
 0  1997                  1.00
 1  1998                  1.01
 2  1999                  1.04
 3  2000                  1.07
 4  2001                  1.08,
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/inflation_multipliers.csv'))

In [ ]:
result